# Tes Permintaan Prediksi — Telco Churn Model

**Author:** andyid 
**Deskripsi:** Mengirim permintaan prediksi sampel ke titik akhir TF Serving REST yang diterapkan dan menampilkan probabilitas churn untuk satu catatan pelanggan.

Pastikan Pelayanan TF berjalan (secara lokal atau di Railway) dan SERVING_URL diatur dengan benar di bawah ini.

In [1]:
import requests
import json
import tensorflow as tf
import base64

# URL Railway Anda
url = "https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict"

# Data dummy untuk dataset Telco Customer Churn (Raw Features)
# PENTING: Semua fitur kategorikal harus dikirim sebagai STRING
raw_data = {
    "gender": "Male",
    "SeniorCitizen": "0",  # <-- HARUS STRING, bukan integer!
    "Partner": "Yes",
    "Dependents": "No",
    "tenure": 12.0,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "Yes",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.7,
    "TotalCharges": 1000.0
}

# Fungsi untuk membuat tf.Example dari raw data
def _create_tf_example(data):
    # === Handle missing value pada TotalCharges ===
    tc_val = data.get('TotalCharges', 0.0)
    if isinstance(tc_val, str) and tc_val.strip() == "":
        tc_val = 0.0  # Ganti dengan 0.0 jika ada missing value (spasi)
        
    feature = {
        # === PERBAIKAN: Tambahkan customerID ===
        # Model mengharuskan fitur ini karena ada di raw schema.
        # Kita ambil dari data, atau beri default 'UNKNOWN' jika tidak ada.
        'customerID': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data.get('customerID', 'UNKNOWN')).encode('utf-8')])),
        
        'gender': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['gender']).encode('utf-8')])),
        
        # SeniorCitizen tetap menggunakan bytes_list (string) sesuai konfirmasi schema.pbtxt Anda
        'SeniorCitizen': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['SeniorCitizen']).encode('utf-8')])), 
        
        'Partner': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['Partner']).encode('utf-8')])),
        'Dependents': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['Dependents']).encode('utf-8')])),
        
        'tenure': tf.train.Feature(int64_list=tf.train.Int64List(value=[int(data['tenure'])])), 
        
        'PhoneService': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['PhoneService']).encode('utf-8')])),
        'MultipleLines': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['MultipleLines']).encode('utf-8')])),
        'InternetService': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['InternetService']).encode('utf-8')])),
        'OnlineSecurity': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['OnlineSecurity']).encode('utf-8')])),
        'OnlineBackup': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['OnlineBackup']).encode('utf-8')])),
        'DeviceProtection': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['DeviceProtection']).encode('utf-8')])),
        'TechSupport': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['TechSupport']).encode('utf-8')])),
        'StreamingTV': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['StreamingTV']).encode('utf-8')])),
        'StreamingMovies': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['StreamingMovies']).encode('utf-8')])),
        'Contract': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['Contract']).encode('utf-8')])),
        'PaperlessBilling': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['PaperlessBilling']).encode('utf-8')])),
        'PaymentMethod': tf.train.Feature(bytes_list=tf.train.BytesList(value=[str(data['PaymentMethod']).encode('utf-8')])),
        
        'MonthlyCharges': tf.train.Feature(float_list=tf.train.FloatList(value=[float(data['MonthlyCharges'])])),
        
        # TotalCharges menggunakan float_list
        'TotalCharges': tf.train.Feature(float_list=tf.train.FloatList(value=[float(tc_val)])), 
    }
    
    tf_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return tf_proto.SerializeToString()


# 1. Serialize data mentah menjadi tf.Example string
serialized_example = _create_tf_example(raw_data)

# 2. TensorFlow Serving REST API mewajibkan bytes/string di-encode ke base64
payload = {
    "instances": [
        {"examples": {"b64": base64.b64encode(serialized_example).decode('utf-8')}}
    ]
}

print(f"🚀 Mengirim request prediksi ke: {url}")
response = requests.post(url, json=payload)

print(f"Status Code: {response.status_code}")
if response.status_code == 200:
    print("✅ Prediksi Berhasil!")
    print(f"Hasil: {response.json()}")
else:
    print("❌ Prediksi Gagal!")
    print(f"Error: {response.text}")

🚀 Mengirim request prediksi ke: https://mlops-deploy.up.railway.app/v1/models/andyid-model:predict
Status Code: 200
✅ Prediksi Berhasil!
Hasil: {'predictions': [[0.642029285]]}
